# Proyecto: Pronóstico de demanda de picking – CEDI
## Notebook 02 · Feature Engineering y control de data leakage

**Equipo:** Natasha Orozco · Xiomy Baquero · Andrés Zabala

### Continuidad del proyecto
El **notebook 01 (EDA)** describió cómo se comporta la demanda de picking (líneas por día) en los canales **Picking Packing** y **TAT**. Sus hallazgos principales fueron:
- El **día de la semana** explica buena parte de la variación en Picking Packing; en TAT pesa más el **nivel reciente**.
- Los **festivos, Semana Santa y el cierre de año** generan caídas y picos que se repiten cada año.
- La serie tiene **cambios de régimen** por decisiones de negocio: expansión de rutas TAT (2024), reducción de catálogo (dic-2025), recorte de rutas TAT (may-2026) y jornada de 42 h (jul-2026).
- Las variables operativas del mismo día (pedidos, SKUs, operarios…) **no se conocen al momento de predecir** y se descartaron (sección 24).

### Objetivo de este notebook
Construir las variables (*features*) con las que el modelo pronosticará las **líneas de picking por día y por canal**, usando **solo información disponible en el corte**.

**Regla del corte:** el pronóstico se hace el **viernes en la noche** para los días operativos de la semana siguiente. Toda variable debe poder calcularse ese viernes.

### Estructura
1. Configuración y carga de datos
2. Calendario operativo
3. Variables de calendario y festivos
4. Variables de eventos
5. Variables de régimen
6. Rezagos e historia reciente (construcción por corte)
7. Carga esperada por rutas de ciudad
8. Validación estadística de cada variable
9. Auditoría de data leakage
10. Tabla final y exportación para el modelado


## 1. Configuración y carga de datos

Se definen las **fechas de régimen** identificadas en el EDA. Son constantes del proyecto: cada variable de régimen se construye a partir de ellas.

| Constante | Fecha | Origen en el EDA |
|---|---|---|
| `EXPANSION_TAT` | 1-abr-2024 | Sección 17: escalón hacia arriba en TAT |
| `CATALOGO_2026` | 22-dic-2025 | Sección 20: quiebre detectado por Zivot-Andrews |
| `RECORTE_RUTAS` | 1-may-2026 | Sección 15: recorte de cobertura TAT |
| `INICIO_42H` | 15-jul-2026 | Ley 2101: jornada de 42 h |

In [1]:
# ============================================
# CELDA 1: Librerías, estilo y constantes del proyecto
# ============================================
!pip install holidays -q
import pandas as pd
import numpy as np
import holidays
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import warnings
warnings.filterwarnings('ignore')

# --- Estilo de las gráficas (Plotly, limpio) ---
pio.templates.default = 'plotly_white'
COL = {'Picking Packing': '#2a78d6', 'TAT': '#eb6834'}   # mismos colores del EDA

# --- Canales ---
CANALES = {'Ruteo Picking Packing': 'Picking Packing', 'Ruteo TAT': 'TAT'}

# --- Fechas de régimen (decisiones de negocio identificadas en el EDA) ---
EXPANSION_TAT  = pd.Timestamp('2024-04-01')   # Sección 17: expansión de rutas TAT (escalón hacia arriba)
CATALOGO_2026  = pd.Timestamp('2025-12-22')   # Sección 20: quiebre Zivot-Andrews, reducción de catálogo
RECORTE_RUTAS  = pd.Timestamp('2026-05-01')   # Sección 15: recorte de cobertura de rutas TAT
INICIO_42H     = pd.Timestamp('2026-07-15')   # Ley 2101: jornada de 42 h, sale el sábado del calendario

# --- Festivos de Colombia (calendario externo, se conoce con años de anticipación) ---
FESTIVOS = pd.to_datetime(list(holidays.Colombia(years=range(2023, 2028)).keys()))

print('Constantes cargadas')
print(f'  Festivos 2023-2027: {len(FESTIVOS)}')
for nombre, fecha in [('Expansión TAT', EXPANSION_TAT), ('Catálogo 2026', CATALOGO_2026),
                      ('Recorte rutas TAT', RECORTE_RUTAS), ('Jornada 42 h', INICIO_42H)]:
    print(f'  {nombre:18s}: {fecha.date()}')

Constantes cargadas
  Festivos 2023-2027: 91
  Expansión TAT     : 2024-04-01
  Catálogo 2026     : 2025-12-22
  Recorte rutas TAT : 2026-05-01
  Jornada 42 h      : 2026-07-15


In [2]:
# ============================================
# CELDA 2: Carga del dataset y validación contra el EDA
# ============================================
RUTA = '/content/dataset_version3.csv'

# Misma lectura del notebook 01 (EDA)
df = pd.read_csv(RUTA, sep=';', decimal=',', encoding='utf-8-sig', low_memory=False)
df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True)
df['Canal'] = df['Canal'].map(CANALES)

# Solo se necesitan 4 columnas: el resto son variables del mismo día (data leakage, sección 24 del EDA)
df = df[['Fecha', 'Canal', 'Ciudad', 'Lineas']]

# Validaciones: si algo no coincide con el EDA, el notebook se detiene aquí
assert len(df) == 35_221,                                     'El número de filas no coincide con el EDA'
assert df['Canal'].notna().all(),                             'Hay canales fuera de PP y TAT'
assert df.isna().sum().sum() == 0,                            'Hay valores nulos'
assert not df.duplicated(['Fecha', 'Canal', 'Ciudad']).any(), 'Hay duplicados Fecha × Canal × Ciudad'
assert (df['Lineas'] > 0).all(),                              'Hay líneas en cero o negativas'

# Serie diaria por canal: la variable objetivo del modelo
diario = df.groupby(['Fecha', 'Canal'], as_index=False)['Lineas'].sum()

print(f'Filas: {len(df):,} | Rango: {df["Fecha"].min().date()} a {df["Fecha"].max().date()}')
print('✓ Validaciones superadas: el dataset es el mismo del EDA\n')
print(diario.groupby('Canal').agg(Dias_con_datos=('Fecha', 'nunique'),
                                  Lineas_totales=('Lineas', 'sum'),
                                  Lineas_dia_mediana=('Lineas', 'median')).to_string())

Filas: 35,221 | Rango: 2023-01-02 a 2026-11-20
✓ Validaciones superadas: el dataset es el mismo del EDA

                 Dias_con_datos  Lineas_totales  Lineas_dia_mediana
Canal                                                              
Picking Packing            1116          778145               699.5
TAT                        1103         2086800              1871.0


### Conclusiones: carga de datos
- El dataset coincide con el del EDA: **35,221 filas**, del 2-ene-2023 al 20-nov-2026, sin nulos ni duplicados.
- **No se exporta un DataFrame limpio desde el EDA:** el feature engineering solo usa `Fecha`, `Canal`, `Ciudad` y `Lineas`, y las validaciones con `assert` garantizan que los datos son los mismos.
- Las demás columnas no se cargan porque son **variables del mismo día** (pedidos, SKUs, operarios…), descartadas por data leakage en la sección 24 del EDA.
- **Serie objetivo:** líneas por día y canal. Mediana de **700 líneas/día en Picking Packing** y **1,871 en TAT**.

## 2. Calendario operativo

**¿Qué días pronostica el modelo?** Solo los días en que el CEDI opera según su calendario base:
- **Lunes a sábado** hasta el 14-jul-2026.
- **Lunes a viernes** desde el 15-jul-2026 (jornada de 42 h).
- Sin **domingos** ni **festivos**.

**Tabla base:** una fila por **día operativo × canal**. Es la estructura sobre la que se construyen todas las variables.

**Dos decisiones heredadas del EDA (sección 17):**
1. Los días operativos **sin datos** quedan como `NaN`, **no como 0**. Un día sin registro no significa demanda cero: en general son sábados de puente festivo o días de Semana Santa en que no se operó.
2. Los días operados **fuera del calendario** (domingos atípicos, el festivo de Reyes de 2026 y los sábados bajo planeación) **no entran** a la tabla base. Su volumen depende de una decisión puntual de operar, no de la demanda normal.

In [3]:
# ============================================
# CELDA 3: Calendario operativo
# ============================================
# 1. Todos los días entre el primero y el último del dataset
fechas = pd.date_range(diario['Fecha'].min(), diario['Fecha'].max(), freq='D')

# 2. Un día es operativo si: no es domingo, no es festivo y no es sábado desde la jornada de 42 h
es_operativo = ((fechas.dayofweek < 6)
                & ~fechas.isin(FESTIVOS)
                & ~((fechas >= INICIO_42H) & (fechas.dayofweek == 5)))
cal_op = fechas[es_operativo]

# 3. Tabla base: una fila por día operativo × canal (NaN = día operativo sin datos, NO se rellena con 0)
base = (pd.MultiIndex.from_product([cal_op, list(COL)], names=['Fecha', 'Canal'])
          .to_frame(index=False)
          .merge(diario, on=['Fecha', 'Canal'], how='left'))

# 4. Días con operación FUERA del calendario: se documentan, no entran a la tabla base
fuera = diario[~diario['Fecha'].isin(cal_op)].copy()
fuera['Motivo'] = np.select(
    [fuera['Fecha'].dt.dayofweek == 6,
     fuera['Fecha'].isin(FESTIVOS),
     (fuera['Fecha'] >= INICIO_42H) & (fuera['Fecha'].dt.dayofweek == 5)],
    ['Domingo', 'Festivo', 'Sábado bajo planeación (42 h)'], 'Otro')

print(f'Días calendario: {len(fechas):,} | Días operativos: {len(cal_op):,}')
print(f'Tabla base: {len(base):,} filas ({len(cal_op):,} días × 2 canales)\n')
print(base.groupby('Canal')['Lineas'].agg(
          Dias_con_datos='count',
          Dias_sin_datos=lambda s: s.isna().sum(),
          Cobertura_pct=lambda s: round(s.notna().mean() * 100, 1)).to_string())
print(f'\nRegistros fuera del calendario operativo: {len(fuera)} (no entran a la tabla base)')
print(fuera.groupby(['Motivo', 'Canal']).agg(Dias=('Fecha', 'count'), Lineas=('Lineas', 'sum')).to_string())

assert len(cal_op) == 1_132, 'El calendario operativo no coincide con el EDA (sección 17)'

Días calendario: 1,419 | Días operativos: 1,132
Tabla base: 2,264 filas (1,132 días × 2 canales)

                 Dias_con_datos  Dias_sin_datos  Cobertura_pct
Canal                                                         
Picking Packing            1108              24           97.9
TAT                        1095              37           96.7

Registros fuera del calendario operativo: 16 (no entran a la tabla base)
                                               Dias  Lineas
Motivo                        Canal                        
Domingo                       Picking Packing     2     650
                              TAT                 2    1900
Festivo                       TAT                 1     483
Sábado bajo planeación (42 h) Picking Packing     6    1447
                              TAT                 5    6558


### Conclusiones: calendario operativo
- **1,132 días operativos** (de 1,419 días calendario) → **tabla base de 2,264 filas** (día × canal).
- **Cobertura alta:** 97.9% en Picking Packing (24 días sin datos) y 96.7% en TAT (37). Los huecos quedan como `NaN` y no se imputan.
- **16 registros fuera del calendario** (12 días distintos) no entran a la tabla base:
  - **Sábados bajo planeación (42 h):** 11 registros con volumen muy variable (de 5 a 2,044 líneas). Dependen de la decisión de operar, no de la demanda normal; si se programa un sábado, se estima aparte.
  - **Domingos atípicos (4) y el festivo de Reyes de 2026 (1):** operación excepcional.
- **Implicación:** el modelo pronostica **días operativos normales**. Los sábados programados quedan como limitación y próximo paso.

> **Nota sobre la granularidad:** el dataset original tiene una fila por **Fecha × Canal × Ciudad** (35,221 filas). La tabla base tiene una fila por **Fecha × Canal** (2,264 filas) porque el objetivo del modelo es el **total de líneas por día y canal**, que es lo que define la planeación de personal. La información por ciudad se conserva en `df` y se usa en la sección 7.

## 3. Variables de calendario y festivos

Todas dependen solo de la fecha, así que **se conocen con anticipación** y no hay riesgo de data leakage.

| Variable | Qué mide | Respaldo en el EDA |
|---|---|---|
| `dia_semana` | 0 = lunes … 5 = sábado | Patrón semanal fuerte en PP (STL: jueves +174, sábado −172 líneas) |
| `mes` | 1 a 12 | Índice estacional: enero −18%, diciembre +14% en PP |
| `dia_antes_festivo` | 1 si el **siguiente día hábil** es festivo | Anomalías alrededor de festivos (sección 18) |
| `dia_despues_festivo` | 1 si el **día hábil anterior** fue festivo | En TAT, el día después de un festivo sale **bajo** |
| `n_festivos_semana` | Festivos que caen en días hábiles de la semana | Un festivo "borra" un día de demanda (sección 7) |
| `dias_operativos_semana` | Días que se operan esa semana | Misma evidencia; separa la caída de Semana Santa |
| `dia_mes`, `es_quincena`, `es_fin_mes` | **Candidatas** del notebook del equipo | Sin respaldo en el EDA: se validan en la sección 8 |


In [4]:
# ============================================
# CELDA 4: Variables de calendario y festivos
# ============================================
def variables_calendario(fechas):
    """Variables que dependen solo de la fecha. Se conocen con anticipación: no hay data leakage."""
    fechas = pd.DatetimeIndex(fechas)

    # Días hábiles "de calendario" ANTES de quitar festivos (lun-sáb, o lun-vie desde la jornada 42 h).
    # Se toma un margen de 14 días para poder mirar el día hábil anterior y el siguiente.
    rango = pd.date_range(fechas.min() - pd.Timedelta(days=14), fechas.max() + pd.Timedelta(days=14))
    habiles = rango[(rango.dayofweek < 6) & ~((rango >= INICIO_42H) & (rango.dayofweek == 5))]

    # Posición de cada fecha dentro de los días hábiles → día hábil anterior y siguiente
    pos = habiles.get_indexer(fechas)
    anterior, siguiente = habiles[pos - 1], habiles[pos + 1]

    # Conteos por semana (lunes a domingo)
    semana_hab = habiles.to_period('W-SUN')
    fest_semana = pd.Series(habiles.isin(FESTIVOS), index=semana_hab).groupby(level=0).sum()
    dias_semana = pd.Series(~habiles.isin(FESTIVOS), index=semana_hab).groupby(level=0).sum()
    semana = fechas.to_period('W-SUN')

    return pd.DataFrame({
        'Fecha':                  fechas,
        'dia_semana':             fechas.dayofweek,                    # 0 = lunes … 5 = sábado
        'mes':                    fechas.month,
        'dia_mes':                fechas.day,
        'es_quincena':            fechas.day.isin([1, 2, 15, 16, 30, 31]).astype(int),   # candidata
        'es_fin_mes':             (fechas.day >= fechas.days_in_month - 2).astype(int),  # candidata
        'dia_antes_festivo':      siguiente.isin(FESTIVOS).astype(int),
        'dia_despues_festivo':    anterior.isin(FESTIVOS).astype(int),
        'n_festivos_semana':      fest_semana.reindex(semana).to_numpy(),
        'dias_operativos_semana': dias_semana.reindex(semana).to_numpy(),
    })

# Agregar las variables a la tabla base
base = base.merge(variables_calendario(cal_op), on='Fecha', how='left')

# --- Revisión rápida ---
print(f'Tabla base: {base.shape[0]:,} filas × {base.shape[1]} columnas\n')
print('Días marcados (por canal):')
for v in ['dia_antes_festivo', 'dia_despues_festivo', 'es_quincena', 'es_fin_mes']:
    print(f'  {v:22s}: {base.loc[base.Canal == "TAT", v].sum()}')
print('\nDías según festivos en la semana:')
print(base[base.Canal == 'TAT'].groupby('n_festivos_semana')['dias_operativos_semana']
      .agg(Dias='size', Dias_operativos_tipicos=lambda s: s.mode()[0]).to_string())

# Ejemplo: semana con lunes festivo (puente de Reyes 2026, 12-ene)
print('\nEjemplo: semana del lunes festivo 12-ene-2026 (Picking Packing)')
ej = base[(base.Canal == 'Picking Packing') & base.Fecha.between('2026-01-08', '2026-01-17')]
print(ej[['Fecha', 'dia_semana', 'Lineas', 'dia_antes_festivo', 'dia_despues_festivo',
          'n_festivos_semana', 'dias_operativos_semana']].to_string(index=False))

Tabla base: 2,264 filas × 12 columnas

Días marcados (por canal):
  dia_antes_festivo     : 63
  dia_despues_festivo   : 63
  es_quincena           : 199
  es_fin_mes            : 113

Días según festivos en la semana:
                   Dias  Dias_operativos_tipicos
n_festivos_semana                               
0                   834                        6
1                   283                        5
2                    12                        4
3                     3                        3

Ejemplo: semana del lunes festivo 12-ene-2026 (Picking Packing)
     Fecha  dia_semana  Lineas  dia_antes_festivo  dia_despues_festivo  n_festivos_semana  dias_operativos_semana
2026-01-08           3   261.0                  0                    0                  0                       6
2026-01-09           4   376.0                  0                    0                  0                       6
2026-01-10           5   121.0                  1                    0          

### Conclusiones: variables de calendario y festivos
- Se agregaron **9 variables** de calendario a la tabla base (2,264 filas × 12 columnas).
- **Festivos bien marcados:** 63 días antes y 63 días después de un festivo. El cálculo contra el **día hábil** (no contra el día calendario) recupera los puentes de lunes festivo, el caso más frecuente en Colombia.
- **Semanas con festivos:** 283 días están en semanas con 1 festivo (5 días operativos) y 15 en semanas de Semana Santa con 2-3 festivos (3-4 días operativos).
- **Ejemplo (puente de Reyes 2026, Picking Packing):** el sábado previo al festivo bajó a 121 líneas, el martes siguiente a 196 y el miércoles subió a 1,187. **El efecto del festivo se reparte en los días de alrededor**, no solo en el día que se pierde.
- `dia_mes`, `es_quincena` y `es_fin_mes` quedan como **candidatas**: el EDA no las respalda. Se conservan solo si la validación estadística (sección 8) muestra que aportan.

## 4. Variables de eventos

Eventos de calendario que **se repiten cada año** y que explicaban 35 de las 48 anomalías del EDA (sección 18). Se conocen con anticipación.

| Variable | Periodo | Respaldo en el EDA |
|---|---|---|
| `semana_santa` | Semana (lun-dom) del Viernes Santo | El miércoles santo casi no se opera |
| `semana_post_semana_santa` | Semana siguiente | Concentra el trabajo represado (10-abr-2026: día récord en TAT) |
| `pre_cierre_anio` | 19 al 23 de diciembre | PP **anticipa**: los mayoristas concentran pedidos antes del cierre |
| `cierre_anio` | 24-dic al 9-ene | TAT **casi se detiene** |
| `reactivacion_anio` | 10 al 17 de enero | Reactivación después del cierre |

**Cambio frente al plan:** el EDA proponía una sola variable `fin_de_anio` (19-dic a 17-ene). Al revisar la mediana por tramos, esa ventana mezcla **efectos opuestos**: el volumen sube del 19 al 23 de diciembre, casi desaparece del 24 de diciembre al 9 de enero y rebota del 10 al 17 de enero. Con una sola variable, el modelo promediaría una subida con una caída y no aprendería ninguna de las dos. Por eso se separa en **tres variables**.

*La Semana Santa se calcula a partir de la fecha de Pascua (`dateutil.easter`), así que también funciona para años futuros.*

In [5]:
# ============================================
# CELDA 5: Variables de eventos
# ============================================
from dateutil.easter import easter

def variables_eventos(fechas):
    """Eventos de calendario que se repiten cada año. Se conocen con anticipación."""
    fechas = pd.DatetimeIndex(fechas)
    semana = fechas.to_period('W-SUN')

    # Semana Santa = semana (lun-dom) que contiene el Viernes Santo (Pascua − 2 días)
    viernes_santo = pd.DatetimeIndex([pd.Timestamp(easter(a)) - pd.Timedelta(days=2)
                                      for a in range(fechas.year.min(), fechas.year.max() + 1)])
    sem_santa = viernes_santo.to_period('W-SUN')

    mes, dia = fechas.month, fechas.day
    return pd.DataFrame({
        'Fecha':                    fechas,
        'semana_santa':             semana.isin(sem_santa).astype(int),
        'semana_post_semana_santa': semana.isin(sem_santa + 1).astype(int),
        'pre_cierre_anio':          ((mes == 12) & dia.isin(range(19, 24))).astype(int),            # 19 al 23 de diciembre
        'cierre_anio':              (((mes == 12) & (dia >= 24)) | ((mes == 1) & (dia <= 9))).astype(int),  # 24-dic al 9-ene
        'reactivacion_anio':        ((mes == 1) & dia.isin(range(10, 18))).astype(int),            # 10 al 17 de enero
    })

base = base.merge(variables_eventos(cal_op), on='Fecha', how='left')

# --- ¿Cómo se comporta el volumen en cada evento? (mediana de líneas vs. días normales) ---
EVENTOS = ['semana_santa', 'semana_post_semana_santa', 'pre_cierre_anio', 'cierre_anio', 'reactivacion_anio']
normal = base[EVENTOS].sum(axis=1).eq(0)
filas = []
for canal in COL:
    b = base[base.Canal == canal]
    ref = b.loc[normal[b.index], 'Lineas'].median()
    filas.append({'Evento': 'Día normal (referencia)', 'Canal': canal,
                  'Dias': int(normal[b.index].sum()), 'Mediana': ref, 'Cambio_%': 0.0})
    for ev in EVENTOS:
        m = b.loc[b[ev] == 1, 'Lineas'].median()
        filas.append({'Evento': ev, 'Canal': canal, 'Dias': int(b[ev].sum()),
                      'Mediana': m, 'Cambio_%': round((m / ref - 1) * 100, 1)})
resumen_eventos = pd.DataFrame(filas)
print(f'Tabla base: {base.shape[0]:,} filas × {base.shape[1]} columnas\n')
print(resumen_eventos.pivot(index='Evento', columns='Canal', values=['Dias', 'Mediana', 'Cambio_%'])
      .reindex(['Día normal (referencia)'] + EVENTOS).to_string())

# --- Gráfica: cambio de volumen por evento ---
fig = px.bar(resumen_eventos[resumen_eventos.Evento != 'Día normal (referencia)'],
             x='Evento', y='Cambio_%', color='Canal', barmode='group', color_discrete_map=COL,
             text='Cambio_%', title='Cambio en la mediana de líneas por día frente a un día normal')
fig.update_traces(texttemplate='%{text:+.0f}%', textposition='outside')
fig.add_hline(y=0, line_color='gray')
fig.update_layout(height=430, width=900, yaxis_title='% vs. día normal', xaxis_title='',
                  legend=dict(orientation='h', y=1.12))
fig.show()

Tabla base: 2,264 filas × 17 columnas

                                    Dias                 Mediana                Cambio_%      
Canal                    Picking Packing     TAT Picking Packing     TAT Picking Packing   TAT
Evento                                                                                        
Día normal (referencia)           1011.0  1011.0           707.5  1868.0             0.0   0.0
semana_santa                        15.0    15.0           301.0  1635.0           -57.5 -12.5
semana_post_semana_santa            24.0    24.0           835.0  2370.5            18.0  26.9
pre_cierre_anio                     13.0    13.0          1159.0  2699.0            63.8  44.5
cierre_anio                         42.0    42.0           194.0   685.5           -72.6 -63.3
reactivacion_anio                   27.0    27.0           725.0  2526.0             2.5  35.2


### Conclusiones: variables de eventos
Cambio en la mediana de líneas por día frente a un día normal (PP 708 · TAT 1,868):

| Evento | Picking Packing | TAT | Lectura |
|---|---|---|---|
| Semana Santa | **−58%** | −13% | PP se frena mucho más que TAT |
| Semana post Semana Santa | +18% | **+27%** | Sale el trabajo represado |
| Pre-cierre (19-23 dic) | **+64%** | **+45%** | Ambos canales anticipan pedidos antes del cierre |
| Cierre (24-dic a 9-ene) | **−73%** | **−63%** | La operación casi se detiene |
| Reactivación (10-17 ene) | +3% | **+35%** | TAT recupera lo represado; PP vuelve a lo normal |

- **Los eventos son los efectos más grandes del calendario:** cambios de −73% a +64%, frente a ±18% de la estacionalidad mensual.
- **Separar el fin de año en tres tramos era necesario:** una sola variable habría mezclado un +64% con un −73%.
- **Los canales reaccionan distinto** (como en el EDA): en Semana Santa PP cae −58% y TAT solo −13%; en la reactivación TAT sube +35% y PP se queda en lo normal. Esto refuerza la decisión de **modelar cada canal por separado**.
- ⚠️ **Pocos días por evento** (13 a 42 en 4 años): el modelo puede sobreajustarse a ellos. Se tendrá en cuenta en la validación (sección 8) y en el modelado.

## 5. Variables de régimen

Indicadoras de **escalón**: valen 0 antes de la decisión de negocio y 1 desde ese día en adelante. Le avisan al modelo que el **nivel** de la demanda cambió por una decisión interna, no por los clientes.

| Variable | Desde | Canal | Efecto esperado (EDA) |
|---|---|---|---|
| `expansion_tat_2024` | 1-abr-2024 | Solo TAT | Escalón hacia arriba: de 48 a ≈65 ciudades por mes |
| `catalogo_2026` | 22-dic-2025 | Ambos | Menos referencias → menos líneas |
| `recorte_rutas_tat` | 1-may-2026 | Solo TAT | Menos tiendas y ciudades atendidas |
| `jornada_42h` | 15-jul-2026 | Ambos | Sale el sábado: el volumen se redistribuye de lunes a viernes |

**¿Por qué son acumulativas?** Hoy las cuatro valen 1 en TAT. Cada una mide el **cambio adicional** que trajo su decisión sobre el nivel que ya existía. Así el modelo separa el efecto de cada una.

**Las variables que son solo de TAT valen 0 en Picking Packing.** Como cada canal se modela por separado, en el modelo de PP serán constantes y no aportan. Así se evita que el modelo de PP le atribuya a una decisión de TAT un cambio que no le corresponde.

**Supuesto sobre el backtesting:** al validar con datos históricos, un corte de abril de 2026 "sabría" que en mayo venía el recorte de rutas, y eso no necesariamente se sabía en su momento.
- La **jornada de 42 h** no tiene este problema: la Ley 2101 de 2021 fijó la fecha con años de anticipación.
- Para el **recorte de rutas** y el **catálogo**, se declara como supuesto: **en producción el modelo siempre conoce el régimen vigente**, porque son decisiones internas que la operación conoce antes de aplicarlas.
- En el modelado se puede medir cuánto cambia el error al quitar estas variables.

In [6]:
# ============================================
# CELDA 6: Variables de régimen
# ============================================
def variables_regimen(fechas, canal):
    """Escalones de nivel por decisiones de negocio. Valen 1 desde la fecha del cambio en adelante."""
    fechas = pd.DatetimeIndex(fechas)
    es_tat = int(canal == 'TAT')
    return pd.DataFrame({
        'Fecha':              fechas,
        'Canal':              canal,
        'expansion_tat_2024': (fechas >= EXPANSION_TAT).astype(int) * es_tat,   # solo TAT
        'catalogo_2026':      (fechas >= CATALOGO_2026).astype(int),            # ambos canales
        'recorte_rutas_tat':  (fechas >= RECORTE_RUTAS).astype(int) * es_tat,   # solo TAT
        'jornada_42h':        (fechas >= INICIO_42H).astype(int),               # ambos canales
    })

regimen = pd.concat([variables_regimen(cal_op, c) for c in COL], ignore_index=True)
base = base.merge(regimen, on=['Fecha', 'Canal'], how='left')

# --- Nivel de cada tramo: mediana de líneas por día (sin días de eventos, para no mezclar efectos) ---
cortes = [base.Fecha.min(), EXPANSION_TAT, CATALOGO_2026, RECORTE_RUTAS, INICIO_42H, base.Fecha.max() + pd.Timedelta(days=1)]
nombres = ['1. Base (ene-23 a mar-24)', '2. Expansión TAT (abr-24 a dic-25)', '3. Catálogo 2026 (dic-25 a abr-26)',
           '4. Recorte rutas TAT (may-26 a jul-26)', '5. Jornada 42 h (jul-26 en adelante)']
base['tramo'] = pd.cut(base.Fecha, bins=cortes, labels=nombres, right=False)
sin_eventos = base[base[EVENTOS].sum(axis=1).eq(0)]
niveles = sin_eventos.pivot_table(index='tramo', columns='Canal', values='Lineas', aggfunc='median', observed=True)
cambio = (niveles.pct_change() * 100).round(1).add_suffix(' cambio %')
print(f'Tabla base: {base.shape[0]:,} filas × {base.shape[1] - 1} columnas (+ tramo, auxiliar para la gráfica)\n')
print(pd.concat([niveles.round(0), cambio], axis=1).to_string())

# --- Gráfica: promedio semanal con los cambios de régimen ---
semanal = (base.dropna(subset=['Lineas'])
               .groupby(['Canal', pd.Grouper(key='Fecha', freq='W-SUN')])['Lineas'].mean().reset_index())
fig = px.line(semanal, x='Fecha', y='Lineas', color='Canal', color_discrete_map=COL,
              title='Líneas por día (promedio semanal) y cambios de régimen')
for fecha, texto in [(EXPANSION_TAT, 'Expansión TAT'), (CATALOGO_2026, 'Catálogo'),
                     (RECORTE_RUTAS, 'Recorte rutas'), (INICIO_42H, 'Jornada 42 h')]:
    fig.add_vline(x=fecha, line_dash='dash', line_color='gray')
    fig.add_annotation(x=fecha, y=1.02, yref='paper', text=texto, showarrow=False, font_size=11)
fig.update_layout(height=450, width=1100, yaxis_title='Líneas por día', xaxis_title='',
                  legend=dict(orientation='h', y=1.12))
fig.show()
base = base.drop(columns='tramo')

Tabla base: 2,264 filas × 21 columnas (+ tramo, auxiliar para la gráfica)

Canal                                   Picking Packing     TAT  Picking Packing cambio %  TAT cambio %
tramo                                                                                                  
1. Base (ene-23 a mar-24)                         633.0  1832.0                       NaN           NaN
2. Expansión TAT (abr-24 a dic-25)                764.0  2019.0                      20.8          10.2
3. Catálogo 2026 (dic-25 a abr-26)                676.0  1814.0                     -11.6         -10.2
4. Recorte rutas TAT (may-26 a jul-26)            642.0  1371.0                      -5.0         -24.4
5. Jornada 42 h (jul-26 en adelante)              797.0  1533.0                      24.1          11.8


### Conclusiones: variables de régimen
Mediana de líneas por día en cada tramo (sin días de eventos):

| Tramo | Picking Packing | TAT |
|---|---|---|
| 1. Base (ene-23 a mar-24) | 633 | 1,832 |
| 2. Expansión TAT (abr-24) | 764 (+21%) | 2,019 (**+10%**) |
| 3. Catálogo 2026 (dic-25) | 676 (**−12%**) | 1,814 (**−10%**) |
| 4. Recorte rutas TAT (may-26) | 642 (−5%) | 1,371 (**−24%**) |
| 5. Jornada 42 h (jul-26) | 797 (**+24%**) | 1,533 (**+12%**) |

- **Cada decisión de negocio deja un escalón visible** y del tamaño esperado: el catálogo baja ≈10-12% en ambos canales, y el recorte de rutas, −24% solo en TAT.
- **La jornada de 42 h sube las líneas por día (+24% PP, +12% TAT) sin que suba la demanda:** el volumen del sábado se reparte de lunes a viernes. Sin esta variable, el modelo confundiría la redistribución con crecimiento.
- **El +21% de PP en el tramo 2 no se debe a la expansión** (que es solo de TAT): es el crecimiento de 2023 a 2024 visto en el EDA (+14%). No es una decisión de negocio identificable, así que lo capturan los **rezagos y el nivel reciente** (sección 6), no una variable de régimen.
- **Régimen actual** (desde el 15-jul-2026): PP ≈ 797 y TAT ≈ 1,533 líneas por día. Es el nivel que el modelo debe acertar en producción.
- **Supuesto declarado:** en el backtesting, las variables de régimen asumen que la decisión se conoce desde que entra en vigencia.

## 6. Rezagos e historia reciente (construcción por corte)

Esta es la sección más delicada del notebook: aquí es donde más fácil se cuela el **data leakage**.

### El corte
El pronóstico se hace al **cierre de la semana** (viernes en la noche desde la jornada de 42 h; sábado en la noche antes) para **todos los días operativos de la semana siguiente**. Por eso:
- Todos los días de una misma semana comparten el **mismo corte**.
- El lunes se pronostica con 1 día de anticipación y el viernes con 5 (`horizonte_dias`).
- **Para pronosticar el jueves no se conoce el miércoles**: el `lag_1` no existe para todo el horizonte.

### Variables
| Variable | Qué es | Respaldo en el EDA (sección 21) |
|---|---|---|
| `lag_7`, `lag_14`, `lag_21`, `lag_28` | Líneas del **mismo día de la semana**, 1 a 4 semanas antes | Correlación de 0.45-0.55 en PP |
| `prom_4_semanas` | Promedio de los cuatro anteriores (ignora los vacíos) | **El mejor predictor** (0.61 en PP) |
| `lag_364` | Mismo día de la semana, **un año antes** | Estacionalidad anual (0.57 en PP) |
| `nivel_reciente` | Promedio de los **últimos 10 días operativos con datos** antes del corte | Reemplaza al `lag_1` sin hacer trampa; clave en TAT |
| `n_lags_disponibles` | Cuántos de los 4 rezagos semanales existen | Indica qué tan confiable es `prom_4_semanas` |


In [7]:
# ============================================
# CELDA 7: Rezagos e historia reciente (construcción por corte)
# ============================================
LAGS = [7, 14, 21, 28, 364]
N_NIVEL = 10          # días operativos para el nivel reciente

def fecha_corte(fechas):
    """Corte = domingo de la semana anterior: el cierre del último día operativo de esa semana
    (viernes desde la jornada de 42 h; sábado antes). Toda la semana objetivo comparte el mismo corte."""
    fechas = pd.DatetimeIndex(fechas)
    return fechas.to_period('W-SUN').start_time - pd.Timedelta(days=1)

def variables_historicas(serie, fechas):
    """serie: líneas por día operativo de UN canal (índice = Fecha, NaN = sin datos).
    Para cada fecha objetivo usa SOLO información hasta su corte."""
    fechas = pd.DatetimeIndex(fechas)
    corte = fecha_corte(fechas)
    out = pd.DataFrame({'Fecha': fechas, 'corte': corte,
                        'horizonte_dias': (fechas - corte).days})

    # 1. Rezagos en días calendario: mismo día de la semana k días antes
    for k in LAGS:
        origen = fechas - pd.Timedelta(days=k)
        assert (origen <= corte).all(), f'lag_{k} usa información posterior al corte'
        out[f'lag_{k}'] = serie.reindex(origen).to_numpy()

    # 2. Promedio del mismo día de la semana en las últimas 4 semanas (ignora los NaN)
    cuatro = [f'lag_{k}' for k in (7, 14, 21, 28)]
    out['prom_4_semanas'] = out[cuatro].mean(axis=1)
    out['n_lags_disponibles'] = out[cuatro].notna().sum(axis=1)

    # 3. Nivel reciente: promedio de los últimos N días operativos CON datos hasta el corte
    observados = serie.dropna()
    nivel = observados.rolling(N_NIVEL, min_periods=5).mean()      # en cada día: promedio de los N últimos
    out['nivel_reciente'] = nivel.reindex(corte, method='ffill').to_numpy()   # último valor ≤ corte
    return out

# Construir por canal sobre la serie del calendario operativo (sin días atípicos)
partes = []
for canal in COL:
    b = base[base.Canal == canal]
    serie = b.set_index('Fecha')['Lineas']
    h = variables_historicas(serie, b['Fecha'])
    h['Canal'] = canal
    partes.append(h)
base = base.merge(pd.concat(partes, ignore_index=True), on=['Fecha', 'Canal'], how='left')

# --- Revisión: disponibilidad de cada variable ---
HIST = [f'lag_{k}' for k in LAGS] + ['prom_4_semanas', 'nivel_reciente']
print(f'Tabla base: {base.shape[0]:,} filas × {base.shape[1]} columnas\n')
print('% de filas con valor disponible:')
print((base.groupby('Canal')[HIST].apply(lambda x: x.notna().mean() * 100)).round(1).T.to_string())

# --- Ejemplo: una semana de TAT y de dónde sale cada variable ---
print('\nEjemplo: semana del 9 al 13 de noviembre de 2026 (TAT) – corte domingo 8-nov')
ej = base[(base.Canal == 'TAT') & base.Fecha.between('2026-11-09', '2026-11-13')]
print(ej[['Fecha', 'Lineas', 'corte', 'horizonte_dias', 'lag_7', 'lag_14', 'lag_364',
          'prom_4_semanas', 'nivel_reciente']].round(0).to_string(index=False))

Tabla base: 2,264 filas × 31 columnas

% de filas con valor disponible:
Canal           Picking Packing   TAT
lag_7                      92.5  91.4
lag_14                     91.8  90.8
lag_21                     91.9  90.7
lag_28                     91.0  89.8
lag_364                    70.1  69.5
prom_4_semanas             99.5  99.5
nivel_reciente             99.5  99.5

Ejemplo: semana del 9 al 13 de noviembre de 2026 (TAT) – corte domingo 8-nov
     Fecha  Lineas      corte  horizonte_dias  lag_7  lag_14  lag_364  prom_4_semanas  nivel_reciente
2026-11-09  1535.0 2026-11-08               1    NaN  1352.0   1923.0          1350.0          1464.0
2026-11-10  2046.0 2026-11-08               2  510.0  1450.0   2781.0          1131.0          1464.0
2026-11-11  1517.0 2026-11-08               3 1429.0  1582.0   2161.0          1473.0          1464.0
2026-11-12  1629.0 2026-11-08               4 1744.0  1837.0   2560.0          1684.0          1464.0
2026-11-13  1380.0 2026-11-08       

### Conclusiones: rezagos e historia reciente
- Se agregaron **9 variables históricas** (31 columnas en total), todas calculadas **solo con información hasta el corte**. El `assert` lo verifica para cada fila.
- **Disponibilidad:** los rezagos semanales existen en el 90-92% de las filas (faltan cuando ese día fue festivo o no tuvo datos). `prom_4_semanas` y `nivel_reciente` cubren el **99.5%**: promediar resuelve los vacíos. `lag_364` cubre el 70% (2023 no tiene año anterior).
- **Ejemplo (TAT, 9-13 nov 2026):** un festivo en la semana anterior deja `lag_7` vacío y el día después del festivo deja un `lag_7` atípico (510 líneas). **Un solo rezago es frágil; el promedio de 4 semanas y el nivel reciente son más estables.**
- **El horizonte va de 1 a 5 días** (de 1 a 6 en el régimen de lunes a sábado). El modelo pronostica la semana completa con la información del cierre de la semana anterior, igual que se haría en la operación.
- El aporte real de cada variable se mide en la **sección 8**.

## 7. Rutas por ciudad: prueba de la variable "carga esperada"

**Hipótesis (EDA, secciones 8 y 12):** la variación diaria se explica en parte por la **programación de rutas**: Apartadó sale los viernes, Copacabana y Barbosa los martes, y Yarumal o Andes 1-2 veces por semana. Además, `lag_14` y `lag_28` superaban a `lag_7` en PP, lo que sugería **rutas quincenales**.

**Variable propuesta: `carga_rutas`.** Para cada ciudad, se estima cuánto aportará el día objetivo con su historia de las **últimas 8 semanas** (mismo día de la semana, siempre antes del corte):
- Ciudad con patrón **quincenal** claro (sale en semanas alternas) → se promedian solo las semanas de la misma paridad.
- Resto de ciudades → se promedian las 8 semanas.
- `carga_rutas` = suma de todas las ciudades.

**Punto clave:** si no hubiera rutas quincenales, sumar los promedios de cada ciudad daría **exactamente el promedio del total** (`prom_8_semanas`). La variable solo aporta si la regla quincenal capta algo que el total no ve. Por eso se compara contra `prom_8_semanas` y contra `prom_4_semanas` (sección 6).

**Criterio:** se incluye solo si mejora la correlación de forma clara, **sobre todo en el régimen actual**, y sin duplicar información.

In [8]:
# ============================================
# CELDA 8: Prueba de la variable "carga esperada por rutas" (ciudad × día)
# ============================================
from scipy.stats import spearmanr

SEMANAS = 8                                   # ventana: últimas 8 semanas antes del corte
calendario = pd.date_range(cal_op.min(), cal_op.max(), freq='D')
prueba = []

for canal in COL:
    b = base[base.Canal == canal].set_index('Fecha')
    # Matriz fecha × ciudad (calendario completo): líneas de cada ciudad cada día; 0 = no se despachó
    M = (df[(df.Canal == canal) & df.Fecha.isin(cal_op)]
           .pivot_table(index='Fecha', columns='Ciudad', values='Lineas', aggfunc='sum')
           .reindex(calendario))
    opero = b['Lineas'].notna().reindex(calendario, fill_value=False)   # el canal operó ese día
    V = M.fillna(0).where(opero, np.nan)                              # NaN si el canal no operó

    # Mismo día de la semana, 1 a 8 semanas antes (siempre ≤ corte): arreglo (semana, fecha, ciudad)
    Vk = np.stack([V.shift(7 * i).to_numpy() for i in range(1, SEMANAS + 1)])
    Dk = np.where(np.isnan(Vk), np.nan, (Vk > 0).astype(float))      # 1 si la ciudad salió ese día
    par = np.arange(1, SEMANAS + 1) % 2 == 0                           # semanas 2, 4, 6, 8 atrás

    with np.errstate(all='ignore'):
        prom_todas = np.nanmean(Vk, axis=0)                            # ciudad semanal
        prom_pares = np.nanmean(Vk[par], axis=0)                       # ciudad quincenal (misma paridad)
        quincenal = np.abs(np.nanmean(Dk[par], axis=0) - np.nanmean(Dk[~par], axis=0)) >= 0.75
    # Carga esperada = suma por ciudad (patrón quincenal claro → solo semanas de la misma paridad)
    carga = np.where(quincenal, prom_pares, prom_todas)
    carga = pd.Series(np.nansum(carga, axis=1), index=calendario).where(
                pd.Series((~np.isnan(Vk[:, :, 0])).sum(axis=0) >= 4, index=calendario))

    # Comparación: promedio de 8 semanas del TOTAL del canal (sin abrir por ciudad)
    prom_8 = pd.Series(np.nanmean(np.stack([b['Lineas'].reindex(calendario).shift(7 * i)
                       for i in range(1, SEMANAS + 1)]), axis=0), index=calendario)

    b = b.assign(carga_rutas=carga.reindex(b.index), prom_8_semanas=prom_8.reindex(b.index)).dropna(subset=['Lineas'])
    for tramo, desde in [('Todo el periodo', b.index.min()), ('Régimen actual (42 h)', INICIO_42H)]:
        x = b[b.index >= desde]
        fila = {'Canal': canal, 'Tramo': tramo}
        for v in ['prom_4_semanas', 'prom_8_semanas', 'carga_rutas']:
            fila[v] = round(spearmanr(x[v], x['Lineas'], nan_policy='omit')[0], 2)
        fila['carga vs prom_8'] = round(spearmanr(x['carga_rutas'], x['prom_8_semanas'], nan_policy='omit')[0], 2)
        prueba.append(fila)

print('Correlación de Spearman con las líneas del día (y entre variables, última columna)\n')
print(pd.DataFrame(prueba).to_string(index=False))

Correlación de Spearman con las líneas del día (y entre variables, última columna)

          Canal                 Tramo  prom_4_semanas  prom_8_semanas  carga_rutas  carga vs prom_8
Picking Packing       Todo el periodo            0.63            0.64         0.67             0.96
Picking Packing Régimen actual (42 h)            0.41            0.29         0.42             0.82
            TAT       Todo el periodo            0.50            0.52         0.54             0.96
            TAT Régimen actual (42 h)            0.07           -0.02        -0.01             1.00


### Conclusiones: rutas por ciudad
| Canal | Tramo | `prom_4_semanas` | `carga_rutas` | Similitud con `prom_8_semanas` |
|---|---|---|---|---|
| PP | Todo el periodo | 0.63 | 0.67 | 0.96 |
| PP | Régimen actual | 0.41 | 0.42 | 0.82 |
| TAT | Todo el periodo | 0.50 | 0.54 | 0.96 |
| TAT | Régimen actual | 0.07 | −0.01 | 1.00 |

**Decisión: `carga_rutas` NO entra al modelo.**
1. **Mejora marginal:** +0.04 de correlación en todo el periodo, y prácticamente nada en el **régimen actual**, que es donde se usará el modelo (PP 0.42 vs. 0.41; TAT sin mejora).
2. **Información duplicada:** se parece en 0.96-1.00 al promedio del total. Abrir por ciudad confirma lo que ya mostraba el EDA (Apartadó los viernes, Copacabana los martes), pero **ese patrón ya está en los rezagos del mismo día de la semana**: si Apartadó sale todos los viernes, el `lag_7` del viernes ya lo trae.
3. **Costo en producción:** exigiría mantener una tabla por ciudad para un beneficio casi nulo.

**Hallazgo adicional:** el promedio de 8 semanas es **peor** que el de 4 en el régimen actual (PP 0.29 vs. 0.41). Una ventana más larga mezcla semanas del régimen anterior; en una serie con cambios de régimen, **la historia reciente pesa más**. Esto confirma `prom_4_semanas` como la ventana adecuada.

**Lo que sí aportaría (mejora futura):** conocer **qué ciudades están programadas** para la semana siguiente según el plan de rutas, antes del corte. Eso no es historia, es información anticipada. No está en el dataset y queda como recomendación para la operación.

## 8. Validación estadística de las variables

**Pregunta:** ¿cada variable aporta información real sobre las líneas del día, o es ruido?

### 8.1 Variables 0/1 de calendario y eventos: prueba de Mann-Whitney

**Mann-Whitney U** compara dos grupos de días (variable = 1 vs. variable = 0) y responde: ¿las líneas de un grupo tienden a ser mayores o menores que las del otro? Se usa en lugar de la prueba t porque las líneas **no tienen distribución normal** (hay días atípicos y colas largas).
- **p < 0.05** → la diferencia difícilmente es casualidad: la variable **aporta**.

**Problema: el efecto escondido del día de la semana.** Una variable puede parecer importante solo porque **coincide** con otra. Por ejemplo, la mayoría de los "días antes de festivo" son **sábados** (antes del lunes festivo), y el sábado ya es el día más bajo de PP. ¿El día es bajo por el festivo o por ser sábado?

**Solución: volumen relativo.**
> `relativo` = líneas del día ÷ `prom_4_semanas` (promedio del mismo día de la semana en las 4 semanas anteriores)

Un valor de **1.00** es un día normal *para ese día de la semana y ese nivel reciente*. Si la variable sigue marcando diferencia en el volumen **relativo**, su efecto es propio y no se debe al día de la semana ni al régimen.

*Las variables de calendario se evalúan sin días de eventos, para no mezclar efectos (lección de la sección 4).*

In [9]:
# ============================================
# CELDA 9: Validación de variables 0/1 de calendario y eventos (Mann-Whitney)
# ============================================
from scipy.stats import mannwhitneyu

# Volumen RELATIVO: líneas del día ÷ promedio del mismo día de la semana en las 4 semanas anteriores.
# 1.00 = un día normal para ese día de la semana; 0.70 = 30% por debajo de lo habitual.
base['relativo'] = base['Lineas'] / base['prom_4_semanas']

BINARIAS = ['dia_antes_festivo', 'dia_despues_festivo', 'es_quincena', 'es_fin_mes'] + EVENTOS
sin_evento = base[EVENTOS].sum(axis=1).eq(0)

filas = []
for canal in COL:
    b = base[(base.Canal == canal) & base.Lineas.notna()]
    for v in BINARIAS:
        # Las variables de calendario se comparan sin días de eventos, para no mezclar efectos
        sub = b if v in EVENTOS else b[sin_evento[b.index]]
        si, no = sub[sub[v] == 1], sub[sub[v] == 0]
        filas.append({
            'Variable': v, 'Canal': canal, 'Dias': len(si),
            'Cambio bruto %': round((si.Lineas.median() / no.Lineas.median() - 1) * 100),
            'p bruto': mannwhitneyu(si.Lineas, no.Lineas).pvalue,
            'Relativo (=1 normal)': round(si.relativo.median(), 2),
            'p relativo': mannwhitneyu(si.relativo.dropna(), no.relativo.dropna()).pvalue})

val_bin = pd.DataFrame(filas)
val_bin['Aporta'] = np.where(val_bin['p relativo'] < 0.05, '✅', '❌')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
for canal in COL:
    print(f'\n=== {canal} ===')
    print(val_bin[val_bin.Canal == canal].drop(columns='Canal').to_string(index=False))


=== Picking Packing ===
                Variable  Dias  Cambio bruto %  p bruto  Relativo (=1 normal)  p relativo Aporta
       dia_antes_festivo    48             -49   0.0000                0.9700      0.5019      ❌
     dia_despues_festivo    48             -31   0.0003                0.7200      0.0000      ✅
             es_quincena   169             -16   0.0022                0.9600      0.0185      ✅
              es_fin_mes   102              -2   0.9205                1.0600      0.2210      ❌
            semana_santa     9             -57   0.0031                0.5700      0.0027      ✅
semana_post_semana_santa    23              19   0.0220                1.1800      0.0041      ✅
         pre_cierre_anio    12              66   0.0000                1.5800      0.0000      ✅
             cierre_anio    33             -73   0.0000                0.2500      0.0000      ✅
       reactivacion_anio    27               3   0.6099                1.1600      0.0058      ✅

=== 

### Conclusiones: variables 0/1 de calendario y eventos

**1. El caso de `dia_antes_festivo`: importante en apariencia, sin efecto real**
- En PP, el día antes de un festivo tiene **49% menos líneas** (p < 0.0001). Parece una variable fuerte.
- Pero en el volumen **relativo** queda en **0.97** (p = 0.50): **no se distingue de un día normal**. La caída se debía a que la mayoría de esos días son **sábados**, el día más bajo de PP. El efecto ya lo captura `dia_semana`.
- **Decisión: se descarta.** Es el ejemplo de por qué hay que controlar las variables que coinciden entre sí.

**2. `dia_despues_festivo`: efecto propio y fuerte**
- Aun controlando el día de la semana, el día después de un festivo queda **28% por debajo de lo normal en PP** (0.72) y **13% en TAT** (0.87), ambos con p < 0.0001. Confirma el hallazgo del EDA: el día después del festivo es **bajo**, no alto. **Se conserva.**

**3. Candidatas del notebook anterior del equipo**
- `es_fin_mes`: **sin efecto** en ningún canal (relativo 1.00-1.06; p > 0.18). **Se descarta.**
- `es_quincena`: efecto **pequeño** (4% por debajo de lo normal) y significativo en ambos canales (p ≈ 0.02). ⚠️ En esta tabla se hacen 18 pruebas; con p < 0.05, por azar se esperaría ~1 falso positivo. **Se conserva como candidata débil** y se confirma en el modelado (si no mejora el error, sale).

**4. Eventos: todos aportan, con diferencias por canal**
- **Cierre de año** (−75% en PP relativo), **pre-cierre** (+58% PP, +25% TAT), **reactivación** (+16% PP, +29% TAT) y **semana post Semana Santa** (+18% PP, +29% TAT) son significativos en ambos canales.
- **Semana Santa** es significativa en PP (−43%), pero **no en TAT** (p = 0.54). Coincide con el EDA: TAT casi no se frena. Se conserva porque cada canal tiene su propio modelo, y en el de TAT puede quedar sin peso.

### 8.2 Variables de régimen: tramo anterior vs. tramo nuevo

Para las variables de régimen **no sirve** la comparación de la celda anterior:
- **Volumen relativo:** `prom_4_semanas` se ajusta al nuevo nivel en 4 semanas, así que el relativo "borra" el escalón. Se usan las líneas brutas.
- **Todos los días con 1 vs. todos con 0:** como las variables son acumulativas, el grupo "1" de `expansion_tat_2024` incluye también la caída de 2026, y la comparación se confunde.

**Prueba correcta:** comparar **el tramo inmediatamente anterior** a la decisión con **el tramo que inicia** (hasta la siguiente decisión), sin días de eventos. Así se aísla el escalón de cada decisión.

In [10]:
# ============================================
# CELDA 10: Validación de variables de régimen (tramo anterior vs. tramo nuevo)
# ============================================
REGIMEN = [('expansion_tat_2024', EXPANSION_TAT, CATALOGO_2026, ['TAT']),
           ('catalogo_2026',      CATALOGO_2026, RECORTE_RUTAS, ['Picking Packing', 'TAT']),
           ('recorte_rutas_tat',  RECORTE_RUTAS, INICIO_42H,    ['TAT']),
           ('jornada_42h',        INICIO_42H,    base.Fecha.max() + pd.Timedelta(days=1), ['Picking Packing', 'TAT'])]
inicios = [base.Fecha.min(), EXPANSION_TAT, CATALOGO_2026, RECORTE_RUTAS, INICIO_42H]

filas = []
for v, desde, hasta, canales in REGIMEN:
    previo = inicios[inicios.index(desde) - 1]          # inicio del tramo anterior
    for canal in canales:
        b = base[(base.Canal == canal) & base.Lineas.notna() & sin_evento]
        antes   = b[(b.Fecha >= previo) & (b.Fecha < desde)].Lineas
        despues = b[(b.Fecha >= desde) & (b.Fecha < hasta)].Lineas
        filas.append({'Variable': v, 'Canal': canal,
                      'Dias antes': len(antes), 'Dias después': len(despues),
                      'Mediana antes': antes.median(), 'Mediana después': despues.median(),
                      'Cambio %': round((despues.median() / antes.median() - 1) * 100, 1),
                      'p': mannwhitneyu(antes, despues).pvalue})
val_reg = pd.DataFrame(filas)
val_reg['Aporta'] = np.where(val_reg['p'] < 0.05, '✅', '❌')
print(val_reg.to_string(index=False))

          Variable           Canal  Dias antes  Dias después  Mediana antes  Mediana después  Cambio %      p Aporta
expansion_tat_2024             TAT         318           466      1832.0000        2019.0000   10.2000 0.0000      ✅
     catalogo_2026 Picking Packing         466            75       764.5000         676.0000  -11.6000 0.0256      ✅
     catalogo_2026             TAT         466            75      2019.0000        1814.0000  -10.2000 0.0002      ✅
 recorte_rutas_tat             TAT          75            57      1814.0000        1371.0000  -24.4000 0.0000      ✅
       jornada_42h Picking Packing          58            87       642.0000         797.0000   24.1000 0.0321      ✅
       jornada_42h             TAT          57            87      1371.0000        1533.0000   11.8000 0.0059      ✅


### Conclusiones: variables de régimen
- **Las 6 combinaciones variable × canal son significativas** (p < 0.05): cada decisión de negocio deja un escalón estadísticamente real.
- **Tamaño de los escalones:** expansión TAT **+10%**, catálogo **−12% PP / −10% TAT**, recorte de rutas **−24% TAT**, jornada 42 h **+24% PP / +12% TAT** (redistribución del sábado).
- El escalón del **catálogo en PP** es el menos contundente (p = 0.026): solo hay 75 días en el tramo nuevo antes del recorte, y parte de la caída de PP es gradual. Se conserva porque coincide con el quiebre de Zivot-Andrews del EDA (22-dic-2025).
- **Decisión: las 4 variables de régimen se conservan.**

### 8.3 Variables numéricas y categóricas

Tres pruebas complementarias:

| Prueba | Para qué variables | Qué responde |
|---|---|---|
| **Spearman** | Numéricas (rezagos, conteos) | ¿Cuando la variable sube, las líneas suben (o bajan)? Va de −1 a +1 |
| **Kruskal-Wallis** | Categóricas (`dia_semana`, `mes`) | ¿La mediana de líneas cambia entre categorías? Es la versión no paramétrica del ANOVA |
| **Información mutua (MI)** | **Todas** | ¿Cuánta información da la variable sobre las líneas, **aunque la relación no sea lineal**? Permite comparar todas en una misma escala |

**Dos tramos:** todo el periodo y el **régimen actual** (desde el 15-jul-2026, 87 días), que es donde se usará el modelo. Una variable que solo funciona en el pasado sirve menos.

*El mes se evalúa con el **índice estacional** (líneas ÷ mediana de su año) en 2023-2025, como en el EDA, para no confundir el mes con los cambios de régimen de 2026.*

In [11]:
# ============================================
# CELDA 11: Validación de variables numéricas y categóricas
# ============================================
from scipy.stats import kruskal
from sklearn.feature_selection import mutual_info_regression

NUMERICAS = ['lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_364', 'prom_4_semanas', 'nivel_reciente',
             'n_lags_disponibles', 'n_festivos_semana', 'dias_operativos_semana', 'dia_mes', 'horizonte_dias']
TRAMOS = [('Todo', base.Fecha.min()), ('Régimen actual', INICIO_42H)]

# --- 1. Spearman: variables numéricas ---
filas = []
for canal in COL:
    for tramo, desde in TRAMOS:
        b = base[(base.Canal == canal) & base.Lineas.notna() & (base.Fecha >= desde)]
        fila = {'Canal': canal, 'Tramo': tramo}
        for v in NUMERICAS:
            rho, p = spearmanr(b[v], b['Lineas'], nan_policy='omit')
            fila[v] = f'{rho:+.2f}' + ('' if p < 0.05 else ' (ns)')      # ns = no significativa
        filas.append(fila)
print('1. CORRELACIÓN DE SPEARMAN CON LAS LÍNEAS  (ns = p ≥ 0.05, no significativa)')
print(pd.DataFrame(filas).set_index(['Canal', 'Tramo']).T.to_string())

# --- 2. Kruskal-Wallis: variables categóricas (¿la mediana cambia entre categorías?) ---
print('\n2. KRUSKAL-WALLIS: DÍA DE LA SEMANA Y MES')
for canal in COL:
    b = base[(base.Canal == canal) & base.Lineas.notna()]
    for tramo, sub in [('Lun-Sáb', b[b.Fecha < INICIO_42H]), ('Lun-Vie (42 h)', b[b.Fecha >= INICIO_42H])]:
        grupos = [g.Lineas for _, g in sub.groupby('dia_semana')]
        med = sub.groupby('dia_semana').Lineas.median()
        print(f'  {canal:16s} dia_semana [{tramo:14s}] p = {kruskal(*grupos).pvalue:.4f} | '
              f'mediana: {med.min():,.0f} (mín) a {med.max():,.0f} (máx)')
    # Mes: índice estacional (líneas ÷ mediana de su año) en 2023-2025, para no mezclar regímenes
    s = b[b.Fecha.dt.year <= 2025].copy()
    s['indice'] = s.Lineas / s.groupby(s.Fecha.dt.year).Lineas.transform('median')
    grupos = [g.indice for _, g in s.groupby('mes')]
    med = s.groupby('mes').indice.median()
    print(f'  {canal:16s} mes        [2023-2025     ] p = {kruskal(*grupos).pvalue:.4f} | '
          f'índice: {med.min():.2f} (mín) a {med.max():.2f} (máx)')

# --- 3. Información mutua: ranking de todas las variables en una misma escala ---
CANDIDATAS = (['dia_semana', 'mes'] + NUMERICAS + BINARIAS +
              ['expansion_tat_2024', 'catalogo_2026', 'recorte_rutas_tat', 'jornada_42h'])
mi = []
for canal in COL:
    b = base[(base.Canal == canal) & base.Lineas.notna()]
    for v in CANDIDATAS:
        x = b[[v, 'Lineas']].dropna()
        if x[v].nunique() < 2:
            continue                                        # constante en este canal (p. ej. recorte en PP)
        valor = mutual_info_regression(x[[v]], x['Lineas'], discrete_features=x[v].nunique() <= 12,
                                       random_state=42)[0]
        mi.append({'Canal': canal, 'Variable': v, 'MI': valor})
mi = pd.DataFrame(mi)
orden = mi.groupby('Variable').MI.max().sort_values().index

fig = px.bar(mi, y='Variable', x='MI', color='Canal', barmode='group', orientation='h',
             color_discrete_map=COL, category_orders={'Variable': list(orden)[::-1]},
             title='Información mutua con las líneas del día (mayor = más información)')
fig.update_layout(height=750, width=900, xaxis_title='Información mutua', yaxis_title='',
                  legend=dict(orientation='h', y=1.05))
fig.show()
print('\n3. TOP 10 POR INFORMACIÓN MUTUA')
print(mi.pivot(index='Variable', columns='Canal', values='MI').round(3)
        .sort_values('Picking Packing', ascending=False).head(10).to_string())

1. CORRELACIÓN DE SPEARMAN CON LAS LÍNEAS  (ns = p ≥ 0.05, no significativa)
Canal                  Picking Packing                        TAT               
Tramo                             Todo Régimen actual        Todo Régimen actual
lag_7                            +0.47          +0.34       +0.34     +0.03 (ns)
lag_14                           +0.56          +0.35       +0.48     +0.10 (ns)
lag_21                           +0.43          +0.23       +0.40     +0.02 (ns)
lag_28                           +0.57          +0.38       +0.47     +0.13 (ns)
lag_364                          +0.55     +0.18 (ns)       +0.22     +0.11 (ns)
prom_4_semanas                   +0.63          +0.41       +0.50     +0.07 (ns)
nivel_reciente                   +0.09     +0.16 (ns)       +0.29     +0.15 (ns)
n_lags_disponibles               +0.21     -0.01 (ns)  +0.04 (ns)     -0.07 (ns)
n_festivos_semana           -0.02 (ns)          +0.22  -0.03 (ns)     -0.01 (ns)
dias_operativos_semana      -0.0


3. TOP 10 POR INFORMACIÓN MUTUA
Canal              Picking Packing    TAT
Variable                                 
dia_semana                  0.3240 0.1200
horizonte_dias              0.3240 0.1200
prom_4_semanas              0.2740 0.1820
lag_14                      0.2150 0.1790
lag_28                      0.2100 0.1070
lag_364                     0.1910 0.0270
lag_21                      0.1360 0.0930
lag_7                       0.0990 0.0960
cierre_anio                 0.0450 0.0290
dia_antes_festivo           0.0330 0.0000


### Conclusiones: variables numéricas y categóricas

**1. Las variables más fuertes (información mutua y Spearman)**
| Variable | PP | TAT | Lectura |
|---|---|---|---|
| `dia_semana` | MI 0.32 · Kruskal p < 0.001 | MI 0.12 · p < 0.001 (Lun-Sáb) | **La más importante en PP** |
| `prom_4_semanas` | ρ = +0.63 (régimen actual +0.41) | ρ = +0.50 (régimen actual +0.07 ns) | **La mejor variable histórica** en ambos canales |
| `lag_14`, `lag_28` | ρ ≈ +0.56 | ρ ≈ +0.48 | Superan a `lag_7` y `lag_21` (patrón quincenal del EDA) |
| `lag_364` | ρ = +0.55 (régimen actual ns) | ρ = +0.22 | Útil en PP con historia; pierde fuerza cuando el año anterior era otro régimen |

**2. TAT en el régimen actual: casi nada es significativo**
- Ningún rezago, ni el día de la semana (Kruskal p = 0.36), explica TAT desde la jornada de 42 h. Confirma el EDA: **el patrón semanal de TAT desapareció** y su variación diaria es casi ruido. **Se debe esperar un error mayor en TAT** que en PP.
- **Excepción: `dia_mes` (+0.54).** Desde mayo de 2026, TAT sube a lo largo del mes (días 1-10: −11%; días 21-31: +10% frente a lo normal). El patrón no existía en 2024-2025 y no se vio en noviembre. **Hipótesis por validar:** empuje comercial de cierre de mes. Se conserva como **candidata para TAT**.

**3. Variables redundantes**
- **`horizonte_dias` = `dia_semana`:** tienen exactamente la misma información mutua (0.324 y 0.120), porque el lunes siempre es horizonte 1, el martes 2, etc. **No entra como variable**: se guarda solo para medir el error por horizonte en el modelado.
- **`n_festivos_semana` y `dias_operativos_semana`** son complementarias (más festivos = menos días operativos). Se conserva solo `dias_operativos_semana`, como candidata débil: aporta en PP en el régimen actual (ρ = −0.22), y el efecto fuerte de Semana Santa ya lo captura su propia variable.

**4. Variables débiles que se conservan por su rol**
- `nivel_reciente`: débil sola en PP (ρ = +0.09) porque no distingue días de la semana, pero **es la referencia de nivel** que el modelo combina con `dia_semana`. En TAT es de las pocas con señal (+0.29). **Se conserva.**
- `n_lags_disponibles`: débil (MI 0.03-0.04), pero indica qué tan confiable es `prom_4_semanas`. **Se conserva** para los modelos de árboles.
- `mes`: estacionalidad clara en TAT (p < 0.001) y marginal en PP (p = 0.054; el efecto del mes en PP se concentra en diciembre-enero, que ya capturan las variables de cierre de año). **Se conserva.**

### 8.4 Multicolinealidad entre variables históricas (VIF)

**Multicolinealidad:** dos o más variables dicen casi lo mismo. En un modelo **lineal** es un problema: el modelo no puede separar el efecto de cada una, los coeficientes se vuelven inestables y pierden interpretación. En los modelos de **árboles** (Random Forest, Gradient Boosting) no afecta la predicción.

**VIF (Variance Inflation Factor):** para cada variable se mide qué tan bien la explican las demás con una regresión lineal.
> VIF = 1 / (1 − R²)

| VIF | Lectura |
|---|---|
| < 5 | Sin problema |
| 5-10 | Moderado |
| > 10 | Severo |
| ∞ | Una variable es **combinación exacta** de otras |

Se comparan tres conjuntos: A (todas las variables históricas), B (rezagos sueltos sin el promedio) y C (promedio + año anterior + nivel reciente).

In [12]:
# ============================================
# CELDA 12: Multicolinealidad entre variables históricas (VIF)
# ============================================
from sklearn.linear_model import LinearRegression

def vif(datos):
    """VIF de cada columna = 1 / (1 − R²), con R² de explicarla con las demás columnas."""
    salida = {}
    for col in datos.columns:
        otras = datos.drop(columns=col)
        r2 = LinearRegression().fit(otras, datos[col]).score(otras, datos[col])
        salida[col] = np.inf if r2 >= 0.9999 else 1 / (1 - r2)
    return pd.Series(salida)

CONJUNTOS = {
    'A. Todas las históricas':            ['lag_7', 'lag_14', 'lag_21', 'lag_28', 'prom_4_semanas', 'lag_364', 'nivel_reciente'],
    'B. Rezagos sueltos (sin promedio)':  ['lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_364', 'nivel_reciente'],
    'C. Promedio + año anterior + nivel': ['prom_4_semanas', 'lag_364', 'nivel_reciente'],
}
tabla = {}
for canal in COL:
    b = base[base.Canal == canal]
    for nombre, cols in CONJUNTOS.items():
        x = b[cols].dropna()                                  # VIF necesita filas completas
        tabla[(canal, nombre[:2])] = vif(x)                   # columna: (canal, 'A.' / 'B.' / 'C.')
tabla = pd.DataFrame(tabla).loc[CONJUNTOS['A. Todas las históricas']]

print('VIF por conjunto de variables  (< 5 bien · 5-10 moderado · > 10 severa · inf = combinación exacta)')
print('A = todas las históricas · B = rezagos sueltos sin promedio · C = promedio + año anterior + nivel\n')
print(tabla.to_string(na_rep='—', float_format=lambda x: f'{x:.1f}'))

# --- Matriz de correlación de Spearman entre variables históricas (PP) ---
cols = CONJUNTOS['A. Todas las históricas']
corr = base[base.Canal == 'Picking Packing'][cols].corr(method='spearman').round(2)
fig = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Correlación entre variables históricas – Picking Packing')
fig.update_layout(height=520, width=620)
fig.show()

VIF por conjunto de variables  (< 5 bien · 5-10 moderado · > 10 severa · inf = combinación exacta)
A = todas las históricas · B = rezagos sueltos sin promedio · C = promedio + año anterior + nivel

               Picking Packing         TAT        
                            A.  B.  C.  A.  B.  C.
lag_7                      inf 1.7   — inf 1.5   —
lag_14                     inf 1.7   — inf 1.4   —
lag_21                     inf 1.5   — inf 1.3   —
lag_28                     inf 1.5   — inf 1.3   —
prom_4_semanas             inf   — 1.5 inf   — 1.4
lag_364                    1.4 1.4 1.5 1.1 1.1 1.1
nivel_reciente             1.2 1.2 1.1 1.7 1.7 1.4


### Conclusiones: multicolinealidad
- **Conjunto A (todas): multicolinealidad perfecta (VIF = ∞).** `prom_4_semanas` es el promedio exacto de los cuatro rezagos. **No se pueden usar juntos en un modelo lineal.**
- **Conjuntos B y C: sin problema (VIF ≤ 1.7).** Aunque los rezagos se correlacionan entre sí (≈0.45-0.55 en PP), ninguno es redundante por sí solo.
- `lag_364` y `nivel_reciente` aportan información **distinta** a los rezagos semanales (VIF 1.1-1.7).

**Decisión según el tipo de modelo:**
| Modelo | Variables históricas | Por qué |
|---|---|---|
| **Lineales** (regresión, Ridge) | Conjunto **C**: `prom_4_semanas` + `lag_364` + `nivel_reciente` | Sin multicolinealidad; el promedio fue la mejor variable histórica (sección 8.3) |
| **Árboles** (Random Forest, Gradient Boosting) | Conjunto **A** completo + `n_lags_disponibles` | La multicolinealidad no afecta su predicción, y los árboles manejan los `NaN` de los rezagos |

### Decisión final: variables para el modelo

#### ✅ Entran al modelo
| Grupo | Variable | PP | TAT | Evidencia principal |
|---|---|---|---|---|
| Calendario | `dia_semana` | ✅ | ✅ | MI 0.32 PP; Kruskal p < 0.001 |
| | `mes` | ✅ | ✅ | Kruskal p < 0.001 TAT; p = 0.054 PP |
| | `dia_despues_festivo` | ✅ | ✅ | Relativo 0.72 PP / 0.87 TAT; p < 0.0001 |
| | `dias_operativos_semana` | ⚠️ | ⚠️ | Débil: ρ = −0.22 en PP (régimen actual) |
| | `es_quincena` | ⚠️ | ⚠️ | Débil: −4%, p ≈ 0.02 (posible falso positivo) |
| | `dia_mes` | — | ⚠️ | Patrón nuevo de 2026 en TAT (ρ = +0.54) |
| Eventos | `semana_santa` | ✅ | ✅* | −43% PP (p = 0.003); *no significativa en TAT |
| | `semana_post_semana_santa` | ✅ | ✅ | +18% PP / +29% TAT |
| | `pre_cierre_anio` | ✅ | ✅ | +58% PP / +25% TAT |
| | `cierre_anio` | ✅ | ✅ | −75% PP / −32% TAT |
| | `reactivacion_anio` | ✅ | ✅ | +16% PP / +29% TAT |
| Régimen | `catalogo_2026` | ✅ | ✅ | −12% PP / −10% TAT |
| | `jornada_42h` | ✅ | ✅ | +24% PP / +12% TAT |
| | `expansion_tat_2024` | — | ✅ | +10% |
| | `recorte_rutas_tat` | — | ✅ | −24% |
| Históricas | Lineales: `prom_4_semanas`, `lag_364`, `nivel_reciente` | ✅ | ✅ | Conjunto C, VIF ≤ 1.5 |
| | Árboles: + `lag_7`, `lag_14`, `lag_21`, `lag_28`, `n_lags_disponibles` | ✅ | ✅ | Conjunto A |

⚠️ = **candidata débil**: entra al primer modelo y sale si no mejora el error (se prueba en el modelado).

**Total:** hasta **20 variables en PP** y **23 en TAT** (modelos de árboles).

#### ❌ Salen del modelo
| Variable | Motivo |
|---|---|
| `dia_antes_festivo` | Su efecto era el del **sábado**: en volumen relativo no se distingue de un día normal (p = 0.50) |
| `es_fin_mes` | Sin efecto en ningún canal (p > 0.18) |
| `n_festivos_semana` | Redundante con `dias_operativos_semana` |
| `horizonte_dias` | Idéntica a `dia_semana`. **Se guarda** solo para medir el error por horizonte |
| `carga_rutas` | Duplica a los rezagos (similitud 0.96-1.00) sin mejorar en el régimen actual (sección 7) |
| `relativo` | Variable **auxiliar** de la validación. **Se calcula con el objetivo (`Lineas`): usarla sería data leakage** |
| `lag_1`, `lag_2` | No están disponibles para todo el horizonte (sección 6) |
| Variables operativas del mismo día | Pedidos, SKUs, operarios, cantidades… (EDA, sección 24) |

#### Expectativa realista para el modelado
- **PP tiene señal clara** (día de la semana + rezagos): se espera un error moderado.
- **TAT en el régimen actual es casi ruido** (ninguna variable histórica es significativa): se espera un error mayor, y el modelo dependerá del nivel y del calendario.

## 9. Auditoría automática de data leakage

La revisión conceptual (filtro 1 de la sección 8) dice qué variables **deberían** ser seguras. Esta auditoría **lo comprueba con el código**.

### Pruebas
| # | Prueba | Qué garantiza |
|---|---|---|
| 0 | **Reproducción:** reconstruir todas las variables desde cero con las funciones de las secciones 3 a 6 y comparar con la tabla base | La tabla que se exporta sale exactamente de las funciones auditadas |
| 1 | **Lista prohibida:** ninguna variable descartada por leakage está en la lista final | Que no se cuele por error una variable del mismo día |
| 2 | **Envenenamiento:** en cada corte semanal, reemplazar por 99,999,999 **(a)** todas las líneas posteriores al corte y **(b)** todas las líneas del otro canal, y recalcular | Si alguna variable cambia, está usando el futuro o mezclando canales → **fuga** |

**La lógica del envenenamiento:** si una variable solo usa información anterior al corte, da igual qué pase después: con datos reales o absurdos, su valor tiene que ser idéntico. Si cambia, está "mirando el futuro".

**Demostración:** para comprobar que la prueba funciona, se audita también la variable `relativo` de la sección 8, que **sí usa las líneas del mismo día**. La auditoría debe detectarla.

In [13]:
# ============================================
# CELDA 13: Auditoría automática de data leakage
# ============================================
# --- Variables finales por canal (decisión de la sección 8) ---
COMUNES = ['dia_semana', 'mes', 'dia_despues_festivo', 'dias_operativos_semana', 'es_quincena',
           'semana_santa', 'semana_post_semana_santa', 'pre_cierre_anio', 'cierre_anio', 'reactivacion_anio',
           'catalogo_2026', 'jornada_42h',
           'lag_7', 'lag_14', 'lag_21', 'lag_28', 'prom_4_semanas', 'lag_364', 'nivel_reciente', 'n_lags_disponibles']
FEATURES = {'Picking Packing': COMUNES,
            'TAT': COMUNES + ['dia_mes', 'expansion_tat_2024', 'recorte_rutas_tat']}
FEATURES_LINEALES = [v for v in COMUNES if v not in ['lag_7', 'lag_14', 'lag_21', 'lag_28', 'n_lags_disponibles']]
PROHIBIDAS = {'Lineas', 'relativo', 'Cantidad', 'Puntos_Envio_Distintos', 'Pedidos_Distintos', 'SKUs_Distintos',
              'Ubicaciones_Visitadas', 'Usuarios_Picking', 'Ruteos_Distintos', 'lag_1', 'lag_2'}

def construir_features(datos, canal, fechas, incluir_relativo=False):
    """Reconstruye TODAS las variables desde cero para unas fechas, con las funciones de las secciones 3-6."""
    fechas = pd.DatetimeIndex(fechas)
    serie = datos[datos.Canal == canal].set_index('Fecha')['Lineas'].reindex(cal_op)   # solo días operativos
    f = (variables_calendario(fechas)
           .merge(variables_eventos(fechas), on='Fecha')
           .merge(variables_regimen(fechas, canal).drop(columns='Canal'), on='Fecha')
           .merge(variables_historicas(serie, fechas), on='Fecha'))
    if incluir_relativo:                       # SOLO para la demostración: usa el objetivo del mismo día
        f['relativo'] = serie.reindex(fechas).to_numpy() / f['prom_4_semanas']
    return f

def auditar(canal, columnas, incluir_relativo=False):
    """Para cada corte semanal: 'envenena' el futuro y el otro canal, y verifica que las variables no cambien."""
    cortes = pd.date_range(cal_op.min() + pd.Timedelta(days=7), cal_op.max(), freq='W-SUN')
    fallas = 0
    for c in cortes:
        semana = cal_op[(cal_op > c) & (cal_op <= c + pd.Timedelta(days=7))]
        if len(semana) == 0:
            continue
        original = construir_features(diario, canal, semana, incluir_relativo)[columnas]
        futuro = diario.copy();  futuro.loc[futuro.Fecha > c, 'Lineas'] = 99_999_999
        otro   = diario.copy();  otro.loc[otro.Canal != canal, 'Lineas'] = 99_999_999
        for alterado in (futuro, otro):
            prueba = construir_features(alterado, canal, semana, incluir_relativo)[columnas]
            if not original.equals(prueba):
                fallas += 1
    return len(cortes), fallas

# --- Prueba 0: las funciones reproducen exactamente la tabla base ---
for canal in COL:
    rehecha = construir_features(diario, canal, cal_op)
    b = base[base.Canal == canal].reset_index(drop=True)
    assert rehecha[FEATURES[canal]].equals(b[FEATURES[canal]]), f'{canal}: la reconstrucción no coincide'
print('Prueba 0 ✓ Las funciones reproducen la tabla base sin diferencias\n')

# --- Prueba 1: ninguna variable prohibida en la lista final ---
for canal in COL:
    assert not PROHIBIDAS & set(FEATURES[canal]), f'{canal}: hay variables prohibidas'
print('Prueba 1 ✓ Ninguna variable prohibida en la lista final\n')

# --- Prueba 2: envenenar el futuro y el otro canal, en TODOS los cortes ---
print('Prueba 2: envenenamiento (futuro = 99,999,999 y otro canal = 99,999,999)')
for canal in COL:
    n, fallas = auditar(canal, FEATURES[canal])
    print(f'  {canal:16s}: {n} cortes auditados · {fallas} fallas  {"✓" if fallas == 0 else "✗ FUGA"}')

# --- Demostración: ¿la auditoría detecta una variable con trampa? ---
n, fallas = auditar('Picking Packing', FEATURES['Picking Packing'] + ['relativo'], incluir_relativo=True)
print(f'\nDemostración con "relativo" (usa las líneas del mismo día): {fallas} fallas en {n} cortes → '
      f'{"✓ la auditoría detecta la fuga" if fallas > 0 else "✗ no la detectó"}')

Prueba 0 ✓ Las funciones reproducen la tabla base sin diferencias

Prueba 1 ✓ Ninguna variable prohibida en la lista final

Prueba 2: envenenamiento (futuro = 99,999,999 y otro canal = 99,999,999)
  Picking Packing : 201 cortes auditados · 0 fallas  ✓
  TAT             : 201 cortes auditados · 0 fallas  ✓

Demostración con "relativo" (usa las líneas del mismo día): 201 fallas en 201 cortes → ✓ la auditoría detecta la fuga


### Conclusiones: auditoría de data leakage
- **Prueba 0 ✓:** las funciones reconstruyen la tabla base exactamente. La tabla que se exporta es la misma que se auditó.
- **Prueba 1 ✓:** ninguna variable descartada por leakage (operativas del mismo día, `lag_1`, `relativo`) está en la lista final.
- **Prueba 2 ✓: 0 fallas en 201 cortes por canal.** Al reemplazar el futuro y el otro canal por valores absurdos, **ninguna variable cambia**: todas usan solo información disponible en el corte y del propio canal.
- **La auditoría funciona:** con `relativo`, una variable que usa las líneas del mismo día, falla en los **201 cortes**. Es decir, si una fuga existiera, se detectaría.
- **Alcance:** esta auditoría certifica la **construcción de variables**. En el modelado hay que mantener la misma disciplina: **validación temporal** (entrenar con el pasado y evaluar con semanas posteriores), nunca una partición aleatoria.

**Variables finales:** 20 en Picking Packing y 23 en TAT para modelos de árboles; 15 en el conjunto para modelos lineales.

## 10. Tabla final y exportación

### Diccionario de variables creadas
**Modelos:** A = árboles (Random Forest, Gradient Boosting) · L = lineales (regresión, Ridge)
**Estado:** ✅ entra · ⚠️ candidata débil (se confirma en el modelado)

| Grupo | Variable | Tipo | Descripción | Valores | Canal | Modelos | Estado |
|---|---|---|---|---|---|---|---|
| **Calendario** | `dia_semana` | Categórica | Día de la semana | 0 = lun … 5 = sáb | Ambos | A, L | ✅ |
| | `mes` | Categórica | Mes del año | 1 a 12 | Ambos | A, L | ✅ |
| | `dia_despues_festivo` | 0/1 | El día hábil anterior fue festivo | 0 / 1 | Ambos | A, L | ✅ |
| | `dias_operativos_semana` | Numérica | Días que se operan en esa semana | 3 a 6 | Ambos | A, L | ⚠️ |
| | `es_quincena` | 0/1 | Días 1, 2, 15, 16, 30, 31 | 0 / 1 | Ambos | A, L | ⚠️ |
| | `dia_mes` | Numérica | Día del mes | 1 a 31 | TAT | A, L | ⚠️ |
| **Eventos** | `semana_santa` | 0/1 | Semana del Viernes Santo | 0 / 1 | Ambos | A, L | ✅ |
| | `semana_post_semana_santa` | 0/1 | Semana siguiente a Semana Santa | 0 / 1 | Ambos | A, L | ✅ |
| | `pre_cierre_anio` | 0/1 | 19 al 23 de diciembre | 0 / 1 | Ambos | A, L | ✅ |
| | `cierre_anio` | 0/1 | 24 de diciembre al 9 de enero | 0 / 1 | Ambos | A, L | ✅ |
| | `reactivacion_anio` | 0/1 | 10 al 17 de enero | 0 / 1 | Ambos | A, L | ✅ |
| **Régimen** | `expansion_tat_2024` | 0/1 escalón | Desde el 1-abr-2024 | 0 / 1 | TAT | A, L | ✅ |
| | `catalogo_2026` | 0/1 escalón | Desde el 22-dic-2025 | 0 / 1 | Ambos | A, L | ✅ |
| | `recorte_rutas_tat` | 0/1 escalón | Desde el 1-may-2026 | 0 / 1 | TAT | A, L | ✅ |
| | `jornada_42h` | 0/1 escalón | Desde el 15-jul-2026 | 0 / 1 | Ambos | A, L | ✅ |
| **Históricas** | `lag_7`, `lag_14`, `lag_21`, `lag_28` | Numérica | Líneas del mismo día de la semana, 1 a 4 semanas antes | Líneas (puede ser NaN) | Ambos | A | ✅ |
| | `prom_4_semanas` | Numérica | Promedio de los 4 rezagos semanales | Líneas | Ambos | A, L | ✅ |
| | `lag_364` | Numérica | Mismo día de la semana, un año antes | Líneas (NaN en 2023) | Ambos | A, L | ✅ |
| | `nivel_reciente` | Numérica | Promedio de los últimos 10 días operativos antes del corte | Líneas | Ambos | A, L | ✅ |
| | `n_lags_disponibles` | Numérica | Cuántos de los 4 rezagos existen | 0 a 4 | Ambos | A | ✅ |

**Columnas que no son variables del modelo:**
| Columna | Rol |
|---|---|
| `Lineas` | **Variable objetivo** |
| `Fecha`, `Canal` | Identifican la fila |
| `corte` | Fecha del corte: sirve para separar entrenamiento y validación **en el tiempo** |
| `horizonte_dias` | Días entre el corte y la fecha: sirve para medir el error por horizonte |

**Totales:** 20 variables en PP y 23 en TAT (modelos de árboles); 15 en PP y 18 en TAT (modelos lineales).

### Archivos exportados
| Archivo | Contenido | Uso |
|---|---|---|
| `features_modelo.csv` | Días con objetivo observado (2023-2026) | Entrenar y validar en el notebook 03 |
| `features_proxima_semana.csv` | Variables de la semana siguiente al último dato (sin objetivo) | Demostrar el pronóstico real en producción |

In [14]:
# ============================================
# CELDA 14: Tabla final y exportación para el modelado
# ============================================
METADATOS = ['Fecha', 'Canal', 'corte', 'horizonte_dias']     # identifican la fila; NO son variables del modelo
OBJETIVO = 'Lineas'
TODAS = list(dict.fromkeys(FEATURES['Picking Packing'] + FEATURES['TAT']))   # unión, sin repetir

# --- 1. Tabla para entrenar: solo días con objetivo observado ---
features_modelo = (base.loc[base[OBJETIVO].notna(), METADATOS + TODAS + [OBJETIVO]]
                       .sort_values(['Canal', 'Fecha']).reset_index(drop=True))
assert not PROHIBIDAS.intersection(set(TODAS)), 'Hay variables prohibidas en la exportación'
assert 'relativo' not in features_modelo.columns, 'La variable auxiliar relativo no debe exportarse'

# --- 2. Variables para la PRÓXIMA semana (uso en producción): corte = último dato disponible ---
corte_actual = diario.Fecha.max().to_period('W-SUN').end_time.normalize()        # domingo de la última semana con datos
proxima = pd.date_range(corte_actual + pd.Timedelta(days=1), periods=7)
proxima = proxima[(proxima.dayofweek < 5) & ~proxima.isin(FESTIVOS)]              # régimen actual: lun-vie
features_proxima = pd.concat([construir_features(diario, c, proxima).assign(Canal=c) for c in COL],
                             ignore_index=True)[METADATOS + TODAS]

features_modelo.to_csv('features_modelo.csv', index=False)
features_proxima.to_csv('features_proxima_semana.csv', index=False)

# --- Resumen ---
print('features_modelo.csv  → entrenamiento y validación')
print(f'  {features_modelo.shape[0]:,} filas × {features_modelo.shape[1]} columnas '
      f'({len(METADATOS)} metadatos + {len(TODAS)} variables + 1 objetivo)')
print(features_modelo.groupby('Canal').agg(Filas=('Fecha', 'size'), Desde=('Fecha', 'min'),
                                           Hasta=('Fecha', 'max')).to_string())
print(f'\nfeatures_proxima_semana.csv → pronóstico real (corte {corte_actual.date()})')
print(features_proxima[['Fecha', 'Canal', 'horizonte_dias', 'dia_semana', 'prom_4_semanas',
                        'nivel_reciente', 'lag_364']].round(0).to_string(index=False))

# En Colab, los archivos se borran al cerrar la sesión: descárgalos o guárdalos en Drive
# from google.colab import files
# files.download('features_modelo.csv'); files.download('features_proxima_semana.csv')

features_modelo.csv  → entrenamiento y validación
  2,203 filas × 28 columnas (4 metadatos + 23 variables + 1 objetivo)
                 Filas      Desde      Hasta
Canal                                       
Picking Packing   1108 2023-01-02 2026-11-20
TAT               1095 2023-01-02 2026-11-20

features_proxima_semana.csv → pronóstico real (corte 2026-11-22)
     Fecha           Canal  horizonte_dias  dia_semana  prom_4_semanas  nivel_reciente   lag_364
2026-11-23 Picking Packing               1           0        898.0000        824.0000  690.0000
2026-11-24 Picking Packing               2           1        545.0000        824.0000  833.0000
2026-11-25 Picking Packing               3           2        969.0000        824.0000  743.0000
2026-11-26 Picking Packing               4           3        837.0000        824.0000  929.0000
2026-11-27 Picking Packing               5           4        745.0000        824.0000  792.0000
2026-11-23             TAT               1          

### Conclusiones: tabla final
- **`features_modelo.csv`:** 2,203 filas (1,108 PP + 1,095 TAT) × 28 columnas: 4 metadatos, 23 variables y el objetivo. Va del 2-ene-2023 al 20-nov-2026.
- **`features_proxima_semana.csv`:** variables de los 5 días operativos del **23 al 27 de noviembre de 2026**, con corte el domingo 22. Demuestra que el flujo funciona en producción: con los datos hasta el viernes, ya se tienen todas las variables de la semana siguiente.
- **Verificado antes de exportar:** ninguna variable prohibida y sin la columna auxiliar `relativo`.

## Conclusiones generales del Feature Engineering

1. **Se construyeron 23 variables** en 4 grupos (calendario, eventos, régimen e historia reciente), todas **disponibles en el corte** del viernes para la semana siguiente.
2. **Los eventos de calendario son los efectos más grandes:** del −73% (cierre de año) al +64% (pre-cierre), frente a ±18% de la estacionalidad mensual.
3. **Cada decisión de negocio dejó un escalón medible:** catálogo −10/−12%, recorte de rutas TAT −24% y jornada 42 h +24% PP / +12% TAT (redistribución del sábado).
4. **La mejor variable histórica es `prom_4_semanas`.** Abrir la información por ciudad no aportó: el patrón de rutas ya está en los rezagos del mismo día de la semana.
5. **Se descartaron 6 variables con evidencia**, entre ellas `dia_antes_festivo`, que parecía fuerte (−49%) pero cuyo efecto era del sábado.
6. **Cero data leakage, comprobado:** 0 fallas en 201 cortes por canal, y la auditoría detecta una fuga cuando se introduce a propósito.
7. **Expectativa para el modelado:** PP tiene señal clara (día de la semana + historia). **TAT en el régimen actual es casi ruido**, así que su error será mayor.

**Hallazgos para validar con la operación:**
- TAT sube a lo largo del mes desde mayo de 2026 (¿empuje comercial de cierre de mes?).
- La información más valiosa que falta: **qué ciudades y pedidos están programados** para la semana siguiente antes del corte.

**Siguiente paso:** notebook 03 (modelado). Se usarán validación temporal por cortes, línea base ingenua (`prom_4_semanas`), modelos lineales y de árboles, y métricas de error por canal y por horizonte.